# Diagnose GPU KV-cache overhead
This is a **short profiling run**, not the 90-run benchmark. It replays 60 hot-prefix requests for each of five strategies, once, using float32, a 32 MiB retained cache, and four output tokens. No Drive storage is needed. Results download at the end.

The model/predictor are unchanged. Timing barriers isolate lookup, prediction, eviction, copying, prefill, and decode. Those barriers alter execution, so these timings must not be presented as benchmark speedups. Prediction is nested inside eviction time: do not add both. The earlier uninstrumented results remain the performance evidence.

Select T4, then Run all. Allow the ZIP download and confirm it is on your laptop before closing the session. Correctness checks remain mandatory. If it fails, send the downloaded diagnostics.


In [ ]:
import os, pathlib, subprocess, sys
os.environ["USE_TF"] = "0"
repo = pathlib.Path("/content/Learning-Based-KV-Cache-Management-for-GPU-LLM-Inference")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/swathiblrs/Learning-Based-KV-Cache-Management-for-GPU-LLM-Inference.git", str(repo)], check=True)
os.chdir(repo)
# Keep Colab's Gradio/Diffusers dependencies intact; reuse its installed CUDA torch.
# virtualenv supplies pip itself; it does not depend on system ensurepip.
subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv>=20.26,<21"], check=True)
subprocess.run([sys.executable, "-m", "virtualenv", "--system-site-packages",
                "--no-download", ".colab-venv"], check=True)
python = str(repo / ".colab-venv/bin/python")
subprocess.run([python, "-m", "pip", "install", "-e", ".[inference]"], check=True)


In [ ]:
subprocess.run([python, "-c", "import torch; assert torch.cuda.is_available(), 'Select an NVIDIA GPU runtime'; print(torch.cuda.get_device_name())"], check=True)

subprocess.run([python, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from google.colab import files
import shutil
import json

run_dir = Path("/content") / ("kv-cache-profile-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f"))
run_dir.mkdir()
archive = str(run_dir) + ".zip"
# Use the frozen initial predictor; do not retrain on benchmark results.
shutil.copy2(repo / "docs/validation/predictor.json", run_dir / "predictor.json")
(run_dir / "code-commit.txt").write_text(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True))
command = [python, "-u", "-m", "kv_cache.cli", "benchmark",
           "--predictor", str(run_dir / "predictor.json"), "--device", "cuda",
           "--dtype", "float32", "--workload", "hot", "--seed", "99",
           "--requests", "60", "--budget-mib", "32", "--new-tokens", "4",
           "--repeats", "1", "--profile-stages", "--output", str(run_dir / "profile.json")]
try:
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    if code:
        raise RuntimeError("Profiling failed: download diagnostics; do not bypass correctness.")
    report = json.loads((run_dir / "profile.json").read_text())
    assert report["complete"] and report["greedy_outputs_match"]
    for row in report["runs"]:
        print("\n", row["policy"], "hits:", row["hits"], "/", row["requests"])
        print("Stage | observed requests | mean ms per observed request")
        for name, stats in row["profiling"]["summary"].items():
            print(f"{name:24s} {stats['calls']:4d} {stats['mean_ms']:.3f}")
    print("Prediction includes features and predictor evaluation, and is already included in eviction_decision.")
finally:
    shutil.make_archive(str(run_dir), "zip", run_dir)
    files.download(archive)


If the browser blocks the download, use the ZIP under `/content` in the Files panel. Do not rerun the profiling cell merely to download again. Upload the ZIP for stage-by-stage analysis. A short trace may exercise some stages only a few times; inspect stage counts before drawing conclusions.
